In [ ]:
# TP 5 géoloc adresses

In [ ]:
# échauffement: construire une URL avec des paramètres

type="rue"
num=3
nom="tourelles"
print(f'https://api-adresse.data.gouv.fr/search/?q={num}+{type}+{nom},Paris&limit=1')



https://api-adresse.data.gouv.fr/search/?q=3+rue+tourelles,Paris&limit=1


In [ ]:
# échauffement: sélectionner des lignes avec iloc et loc

# iloc sélectionne par rapport "au nombre" de lignes 
# de manière indépendante de l'index 
# alors que loc sélectionne par rapport à l'index

import pandas as pd

df = pd.DataFrame({
    "number": [3, 18, 7, 12, 5],
    "name": ["tourelles", "BERNARDINS", "Bouvart", "Rivoli", "Seine"],
})
sub = df[df.number > 10]

# 1.
# sub contient df avec uniquement les lignes où
# le "number" était supérieur à 10

# 2.
print(df.iloc[:2])
print(df.loc[:2])
# donc loc prend la ligne numéro 2 incluse dans l'Index


   number        name
0       3   tourelles
1      18  BERNARDINS
   number        name
0       3   tourelles
1      18  BERNARDINS
2       7     Bouvart


In [ ]:
# 3.

print(sub.iloc[:2])
# On a pris les deux premières lignes 
# de manière indépendantes des valeurs de l'index

# Là on prend juste la ligne dont le numéro 
# dans l'index est <2
print(sub.loc[:2])


   number        name
1      18  BERNARDINS
3      12      Rivoli
   number        name
1      18  BERNARDINS


In [ ]:
# 4.

print(sub.iloc[0])
# print(sub.loc[0]) renvoie une erreur
# c'est normal on a enlevé la ligne d'indexe 0


number            18
name      BERNARDINS
Name: 1, dtype: object


In [11]:
# 5.

print(sub.iloc[0]["name"])
print(sub.loc[1]["name"])

BERNARDINS
BERNARDINS


In [13]:
# géolocalisation

import pandas as pd

# load the data in data/addresses.csv
# and display a few first lines

addresses = pd.read_csv('data 5/addresses.csv')
addresses.head(4)


,number,type,name,city
0,5,rue,RIQUET,Paris
1,141,boulevard,ITALIE,Paris
2,234,rue,FAUBOURG ST-HONORÉ,Paris
3,62,rue,MICHEL BIZOT,Paris


In [ ]:
adresses_small = addresses.iloc[:20]

adresses_small.to_csv("data 5/addresses-small.csv", index=False)


In [27]:
# une par une 

#&limit=1 permet de limiter la liste/dictionnaire au premier

import requests
import time


def localize_one(num,typ,nom):
    response=requests.get(f'https://api-adresse.data.gouv.fr/search/?q={num}+{typ}+{nom},Paris&limit=1')
    if not ((response.status_code >= 200) and (response.status_code<300)):
        return None
    return (response.json())



localize_one(5, 'square', 'Trocadéro')



{'type': 'FeatureCollection',
 'features': [{'type': 'Feature',
   'geometry': {'type': 'Point', 'coordinates': [2.282475, 48.862707]},
   'properties': {'label': '5 Square du Trocadéro 75116 Paris',
    'score': 0.7820780419580419,
    'housenumber': '5',
    'id': '75116_9453_00005',
    'banId': '5cb1f272-d640-4887-9f4e-2f2b62406b56',
    'name': '5 Square du Trocadéro',
    'postcode': '75116',
    'citycode': '75116',
    'x': 647359.4,
    'y': 6862758.53,
    'city': 'Paris',
    'district': 'Paris 16e Arrondissement',
    'context': '75, Paris, Île-de-France',
    'type': 'housenumber',
    'importance': 0.64132,
    'depcode': '75',
    'street': 'Square du Trocadéro',
    '_type': 'address'}}],
 'query': '5 square Trocadéro,Paris'}

In [31]:
# en un seul coup

# requests is the swiss knife for doing http

import requests
# this one is to fake a file from a string
import io

def mass_post(filename, col_number, col_type, col_name, col_city):

    # we don't care about the index for now
    # but note that if we want to merge the result
    # with the initial data, there is a need 
    # for a unique index...

    with open(filename) as feed:
        response = requests.post(
            "https://api-adresse.data.gouv.fr/search/csv/", 
            files={'data': feed},
            data={'columns': [col_number, col_type, col_name, col_city]})
    if not (200 <= response.status_code < 300):
        print(f"OOPS, got {response.status_code}")
        return None

    # this time the output is csv, not json
    # note that read_csv requires a file-like object
    # and we have a str, so io.StringIO comes to the rescue
    result = pd.read_csv(io.StringIO(response.text))

    return result


def localize_many(filename, col_number, col_type, col_name, col_city):
    df=pd.read_csv(filename,encoding='utf-8')
    geoloc=mass_post(filename,col_number,col_type,col_name,col_city)
    geoloc=geoloc[['latitude','longitude','result_city','result_type']]
    return df.merge(geoloc,left_index=True,right_index=True)

addresses_small = localize_many("data 5/addresses-small.csv", "number", "type", "name", "city")
addresses_small


,number,type,name,city,latitude,longitude,result_city,result_type
0,5,rue,RIQUET,Paris,48.888134,2.376448,Paris,housenumber
1,141,boulevard,ITALIE,Paris,46.669103,-1.418872,La Roche-sur-Yon,street
2,234,rue,FAUBOURG ST-HONORÉ,Paris,48.876160,2.302766,Paris,housenumber
3,62,rue,MICHEL BIZOT,Paris,48.835423,2.402040,Paris,housenumber
4,6,rue,OURS,Paris,48.863311,2.351775,Paris,housenumber
5,30,rue,PETIT-MUSC,Paris,48.852945,2.364722,Paris,housenumber
6,3,impasse,MASSONET,Paris,48.895498,2.351580,Paris,housenumber
7,56,rue,PANOYAUX,Paris,48.867023,2.388347,Paris,housenumber
8,16,rue,THÉRÈSE,Paris,48.866374,2.335212,Paris,housenumber
9,11,rue,JEAN ROBERT,Paris,48.889520,2.358170,Paris,housenumber


In [ ]:
bad=addresses_small[addresses_small.result_city.fillna("")!="Paris"]


In [ ]:
# Afficher sur une carte 

import folium

# ~ chatelet

CENTER = 48.856542, 2.347614

In [39]:
# le fond de la carte

import pandas as pd
addresses = pd.read_csv("data 5/addresses-geoloc.csv")
addresses_small = addresses.iloc[:20]

CENTER = 48.856542, 2.347614

def paris_map():
    m = folium.Map(location=(CENTER),zoom_start=13)
    return (m)


paris_map()
